# HW0. Your first TPU run

This Jupyter notebook has one Cloud TPU v5e chip attached for the entire session.
Run the cells directly; no job submission or queue is needed. A chip remains allocated
while your notebook server is running, even when you are only reading or editing.
Stop your server when you finish. Inactive servers are stopped automatically after
30 minutes, and every server has an eight-hour maximum age.


In [ ]:
import jax

devices = jax.devices()
print("devices:", devices)
assert devices and all(device.platform == "tpu" for device in devices), devices


You should see one `TpuDevice`. If this cell fails, tell a TA before continuing.

## Part 1. Attention, and where the time goes

Below is a single multi-head attention forward pass, the shape that dominates an LLM
forward pass. Run it, then answer the questions underneath.


In [ ]:
import time
import jax.numpy as jnp

B, H, S, D = 8, 12, 1024, 64
key = jax.random.PRNGKey(0)
q, k, v = jax.random.normal(key, (3, B, H, S, D), dtype=jnp.bfloat16)

@jax.jit
def attn(q, k, v):
    logits = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(D).astype(q.dtype)
    return jnp.einsum("bhqk,bhkd->bhqd", jax.nn.softmax(logits, axis=-1), v)

attn(q, k, v).block_until_ready()          # compile first, time second
t0 = time.perf_counter()
for _ in range(50):
    out = attn(q, k, v)
out.block_until_ready()
dt = (time.perf_counter() - t0) / 50

flops = 4 * B * H * S * S * D
print(f"{dt*1e3:.3f} ms per pass")
print(f"{flops/dt/1e12:.2f} TFLOP/s")


**Q1.** A v5e chip peaks near 197 TFLOP/s in bfloat16. You measured far less than that.
Work out how many bytes the `logits` array occupies and how many bytes the chip must
move to compute the softmax over it. Is this kernel limited by arithmetic or by memory
bandwidth? Show the arithmetic intensity you calculated.

**Q2.** Double `S` to 2048 and run again. Predict the change in time *before* you run
it, then explain any gap between your prediction and the measurement.

## Part 2. Make it faster

The two-einsum version above materialises the whole `B × H × S × S` logits array in
memory. Flash attention avoids that by tiling over the sequence and keeping a running
softmax, so the large intermediate never exists.

Use `jax.nn.dot_product_attention` and compare. Report both numbers and explain the
difference in terms of the bytes moved, not just the wall time.

In [ ]:
# Your code here. It runs directly on the TPU attached to this notebook.


## Finishing your session

Save your notebook with its outputs, then click the red Stop TPU Server button
in JupyterLab. Closing the browser tab alone may leave the TPU allocated until the idle
culler stops the server.

## Submitting

Hand in this notebook with all outputs kept. The first cell records the TPU device
used for the measurements.
